<a href="https://colab.research.google.com/github/samyuktha07-prog/voice-based-viva-/blob/main/voicebasedviva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 🎓 VOICE-BASED VIVA EXAMINER
# TEXT & SPEECH ANALYSIS MINI PROJECT
# GOOGLE COLAB - SINGLE CELL
# ============================================================

!pip -q install gradio pandas matplotlib

import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import re
import random
import time

# ============================================================
# QUESTION BANK
# ============================================================

QUESTION_BANK = {

    "Python": [
        {
            "q": "What is Python?",
            "keywords": ["programming", "language", "interpreted", "high level"]
        },
        {
            "q": "What is a Python list?",
            "keywords": ["collection", "ordered", "mutable", "elements"]
        },
        {
            "q": "What is a function in Python?",
            "keywords": ["function", "def", "reusable", "code", "parameters"]
        },
        {
            "q": "What is the difference between a list and tuple?",
            "keywords": ["list", "tuple", "mutable", "immutable"]
        }
    ],

    "Cloud Computing": [
        {
            "q": "What is cloud computing?",
            "keywords": ["internet", "resources", "servers", "storage", "computing"]
        },
        {
            "q": "What are the advantages of cloud computing?",
            "keywords": ["scalability", "cost", "availability", "flexibility"]
        },
        {
            "q": "What is SaaS?",
            "keywords": ["software", "service", "internet", "application"]
        },
        {
            "q": "What is virtualization?",
            "keywords": ["virtual", "machine", "hardware", "resources"]
        }
    ],

    "Computer Networks": [
        {
            "q": "What is a computer network?",
            "keywords": ["devices", "communication", "data", "connection"]
        },
        {
            "q": "What is an IP address?",
            "keywords": ["address", "device", "network", "communication"]
        },
        {
            "q": "What is TCP?",
            "keywords": ["transport", "connection", "reliable", "protocol"]
        },
        {
            "q": "What is the difference between TCP and UDP?",
            "keywords": ["tcp", "udp", "connection", "reliable", "speed"]
        }
    ],

    "Artificial Intelligence": [
        {
            "q": "What is Artificial Intelligence?",
            "keywords": ["machine", "intelligence", "human", "tasks", "learning"]
        },
        {
            "q": "What is Machine Learning?",
            "keywords": ["data", "learning", "model", "prediction", "algorithm"]
        },
        {
            "q": "What is NLP?",
            "keywords": ["language", "text", "computer", "human", "processing"]
        },
        {
            "q": "What is deep learning?",
            "keywords": ["neural", "network", "layers", "learning", "data"]
        }
    ],

    "Text and Speech Analysis": [
        {
            "q": "What is speech recognition?",
            "keywords": ["speech", "text", "audio", "recognition", "conversion"]
        },
        {
            "q": "What is Text-to-Speech?",
            "keywords": ["text", "speech", "audio", "synthesis", "voice"]
        },
        {
            "q": "What is sentiment analysis?",
            "keywords": ["text", "emotion", "positive", "negative", "sentiment"]
        },
        {
            "q": "What is Natural Language Processing?",
            "keywords": ["language", "text", "computer", "processing", "human"]
        }
    ]
}


# ============================================================
# GLOBAL SESSION DATA
# ============================================================

session = {
    "questions": [],
    "current": 0,
    "scores": [],
    "answers": [],
    "started": False
}


# ============================================================
# TEXT UTILITIES
# ============================================================

def words(text):

    return re.findall(
        r"\b[a-zA-Z]+\b",
        text.lower()
    )


def clean(text):

    if text is None:
        return ""

    text = str(text)

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# ============================================================
# KEYWORD MATCHING
# ============================================================

def keyword_analysis(answer, keywords):

    answer_words = words(answer)

    answer_text = " ".join(answer_words)

    found = []
    missing = []

    for key in keywords:

        key_words = key.lower().split()

        if all(
            word in answer_text
            for word in key_words
        ):

            found.append(key)

        else:

            missing.append(key)

    if len(keywords) == 0:

        score = 0

    else:

        score = (
            len(found)
            / len(keywords)
            * 100
        )

    return found, missing, round(score, 2)


# ============================================================
# ANSWER LENGTH ANALYSIS
# ============================================================

def length_score(answer):

    count = len(words(answer))

    if count >= 80:

        return 100

    if count >= 50:

        return 90

    if count >= 30:

        return 80

    if count >= 15:

        return 65

    return 40


# ============================================================
# FILLER ANALYSIS
# ============================================================

FILLERS = [
    "um",
    "uh",
    "like",
    "basically",
    "actually",
    "you know",
    "i mean",
    "literally"
]


def filler_analysis(answer):

    text = answer.lower()

    total = 0
    details = {}

    for filler in FILLERS:

        pattern = r"\b" + re.escape(filler) + r"\b"

        count = len(
            re.findall(
                pattern,
                text
            )
        )

        if count > 0:

            details[filler] = count

            total += count

    word_count = len(words(answer))

    if word_count == 0:

        rate = 0

    else:

        rate = total / word_count

    if rate <= 0.02:

        score = 100

    elif rate <= 0.05:

        score = 90

    elif rate <= 0.08:

        score = 75

    else:

        score = 55

    return details, total, score


# ============================================================
# SENTIMENT / CONFIDENCE
# ============================================================

POSITIVE = [
    "confident",
    "good",
    "great",
    "strong",
    "successful",
    "understand",
    "learn",
    "improve",
    "experience",
    "knowledge",
    "achieve"
]

NEGATIVE = [
    "don't know",
    "cannot",
    "unable",
    "confused",
    "difficult",
    "bad",
    "weak"
]


def tone_analysis(answer):

    text = answer.lower()

    positive_count = sum(
        text.count(x)
        for x in POSITIVE
    )

    negative_count = sum(
        text.count(x)
        for x in NEGATIVE
    )

    if positive_count > negative_count:

        return "Confident / Positive 😊", 90

    if negative_count > positive_count:

        return "Uncertain / Negative 😟", 55

    return "Neutral 😐", 75


# ============================================================
# ANSWER EVALUATION
# ============================================================

def evaluate_answer(answer, question_data):

    answer = clean(answer)

    if not answer:

        return None

    keywords = question_data["keywords"]

    found, missing, keyword_score = keyword_analysis(
        answer,
        keywords
    )

    length = length_score(answer)

    filler_details, filler_count, filler_score = filler_analysis(
        answer
    )

    tone, tone_score = tone_analysis(answer)

    # Final weighted score
    score = (
        keyword_score * 0.45
        + length * 0.20
        + filler_score * 0.15
        + tone_score * 0.20
    )

    score = round(score, 2)

    if score >= 90:

        level = "Excellent 🏆"

    elif score >= 80:

        level = "Very Good 🌟"

    elif score >= 70:

        level = "Good 👍"

    elif score >= 60:

        level = "Average 🙂"

    else:

        level = "Needs Improvement 📚"

    strengths = []

    improvements = []

    if keyword_score >= 70:

        strengths.append(
            "Good coverage of important concepts."
        )

    else:

        improvements.append(
            "Include more important technical concepts."
        )

    if length >= 80:

        strengths.append(
            "Answer has good explanation depth."
        )

    else:

        improvements.append(
            "Explain the answer with an example or additional details."
        )

    if filler_score >= 90:

        strengths.append(
            "Good control of filler words."
        )

    else:

        improvements.append(
            "Reduce filler words and unnecessary repetition."
        )

    if tone_score >= 85:

        strengths.append(
            "Answer shows a confident communication style."
        )

    else:

        improvements.append(
            "Try to answer more confidently and directly."
        )

    return {
        "score": score,
        "level": level,
        "found": found,
        "missing": missing,
        "keyword_score": keyword_score,
        "length_score": length,
        "filler_score": filler_score,
        "filler_count": filler_count,
        "filler_details": filler_details,
        "tone": tone,
        "tone_score": tone_score,
        "strengths": strengths,
        "improvements": improvements
    }


# ============================================================
# START VIVA
# ============================================================

def start_viva(topic, number):

    global session

    number = int(number)

    bank = QUESTION_BANK[topic]

    count = min(
        number,
        len(bank)
    )

    session = {
        "questions": random.sample(
            bank,
            count
        ),
        "current": 0,
        "scores": [],
        "answers": [],
        "started": True
    }

    first = session["questions"][0]

    return (
        f"# 🎓 Viva Started\n\n"
        f"### Question 1 of {count}\n\n"
        f"## {first['q']}",
        "Ready! 🎤",
        "",
        ""
    )


# ============================================================
# SUBMIT ANSWER
# ============================================================

def submit_answer(answer):

    global session

    if not session["started"]:

        return (
            "⚠️ Start the viva first.",
            "",
            "",
            "",
            ""
        )

    answer = clean(answer)

    if not answer:

        return (
            "⚠️ Please enter your answer.",
            "",
            "",
            "",
            ""
        )

    current_question = session["questions"][
        session["current"]
    ]

    result = evaluate_answer(
        answer,
        current_question
    )

    session["scores"].append(
        result["score"]
    )

    session["answers"].append({
        "question": current_question["q"],
        "answer": answer,
        "score": result["score"]
    })

    strengths = "\n".join(
        "✅ " + x
        for x in result["strengths"]
    )

    improvements = "\n".join(
        "🔸 " + x
        for x in result["improvements"]
    )

    feedback = f"""
## 📊 Answer Evaluation

### Score: {result['score']}/100

**Performance:** {result['level']}

**Tone:** {result['tone']}

### 🔑 Concepts Found
{", ".join(result['found']) if result['found'] else "None"}

### 📚 Concepts Missing
{", ".join(result['missing']) if result['missing'] else "None"}

### 💪 Strengths
{strengths}

### 🚀 Improve
{improvements}
"""

    # Move to next question
    session["current"] += 1

    if session["current"] >= len(session["questions"]):

        next_question = (
            "## 🎉 Viva Completed!\n\n"
            "Click **Generate Final Report** to see your complete performance."
        )

    else:

        qnum = session["current"] + 1

        next_question = (
            f"## 🎤 Question {qnum} "
            f"of {len(session['questions'])}\n\n"
            f"### {session['questions'][session['current']]['q']}"
        )

    return (
        feedback,
        next_question,
        "",
        f"Question {session['current']} completed.",
        result["score"]
    )


# ============================================================
# NEXT QUESTION
# ============================================================

def next_question():

    global session

    if not session["started"]:

        return "⚠️ Start the viva first."

    if session["current"] >= len(session["questions"]):

        return "🎉 All questions completed!"

    q = session["questions"][
        session["current"]
    ]

    return (
        f"## 🎤 Question "
        f"{session['current'] + 1} "
        f"of {len(session['questions'])}\n\n"
        f"### {q['q']}"
    )


# ============================================================
# FINAL REPORT
# ============================================================

def final_report():

    global session

    if not session["scores"]:

        return (
            "⚠️ No answers have been evaluated yet.",
            None
        )

    scores = session["scores"]

    average = round(
        sum(scores) / len(scores),
        2
    )

    highest = max(scores)

    lowest = min(scores)

    if average >= 90:

        grade = "A+ 🏆"

    elif average >= 80:

        grade = "A 🌟"

    elif average >= 70:

        grade = "B 👍"

    elif average >= 60:

        grade = "C 🙂"

    else:

        grade = "D 📚"

    rows = []

    for i, item in enumerate(
        session["answers"],
        1
    ):

        rows.append({
            "Question": f"Q{i}",
            "Score": item["score"],
            "Question Text": item["question"]
        })

    df = pd.DataFrame(rows)

    report = f"""
# 🎓 FINAL VIVA REPORT

## 🏆 Overall Score

# {average}/100

### Grade: {grade}

---

## 📊 Performance Summary

**Questions Answered:** {len(scores)}

**Highest Score:** {highest}

**Lowest Score:** {lowest}

**Average Score:** {average}

---

## 💡 Overall Feedback

"""

    if average >= 85:

        report += (
            "Excellent viva performance! "
            "Your answers show strong conceptual understanding. "
            "Continue practicing real-world examples."
        )

    elif average >= 70:

        report += (
            "Good performance. "
            "Your fundamentals are developing well. "
            "Focus on explaining concepts with examples."
        )

    elif average >= 60:

        report += (
            "Average performance. "
            "Revise the important concepts and practice "
            "explaining answers aloud."
        )

    else:

        report += (
            "More preparation is recommended. "
            "Review the fundamentals and practice short "
            "technical explanations."
        )

    report += """

---

## 📌 Viva Preparation Tips

1. Understand concepts instead of memorizing definitions.
2. Give a simple example whenever possible.
3. Speak clearly and confidently.
4. Structure answers logically.
5. If you don't know something, explain what you do know.
6. Practice technical questions aloud.

---

### 🚀 Voice-Based Viva Examiner

**Text & Speech Analysis Mini Project**

Technology: Python + Gradio + NLP-based Text Analysis
"""

    return (
        report,
        df
    )


# ============================================================
# SAMPLE ANSWER
# ============================================================

def sample_answer():

    return (
        "Python is a high level interpreted programming language. "
        "It is widely used because its syntax is simple and easy "
        "to understand. Python supports different programming "
        "paradigms and provides many libraries. It is commonly "
        "used for web development, data analysis, artificial "
        "intelligence and automation."
    )


# ============================================================
# RESET
# ============================================================

def reset_viva():

    global session

    session = {
        "questions": [],
        "current": 0,
        "scores": [],
        "answers": [],
        "started": False
    }

    return (
        "### 🎓 Viva not started",
        "",
        "",
        "",
        None
    )


# ============================================================
# CSS
# ============================================================

css = """
.gradio-container {
    max-width: 1150px !important;
    margin: auto !important;
}

.title {
    text-align: center;
    padding: 25px;
    border-radius: 20px;
    background: linear-gradient(135deg, #667eea, #764ba2);
    color: white;
}

.card {
    border-radius: 15px;
}
"""


# ============================================================
# GRADIO APP
# ============================================================

with gr.Blocks(
    title="Voice-Based Viva Examiner",
    css=css
) as app:

    gr.HTML("""
    <div class="title">
        <h1>🎓 Voice-Based Viva Examiner</h1>
        <h3>AI-Powered Text & Speech Analysis Application</h3>
        <p>
        Practice viva questions, analyze answers,
        receive instant feedback and generate a final report.
        </p>
    </div>
    """)

    gr.Markdown(
        """
        ### 🚀 How it works

        **Select Subject → Start Viva → Answer Question →
        AI Evaluation → Next Question → Final Report**
        """
    )

    with gr.Row():

        with gr.Column(scale=1):

            topic = gr.Dropdown(
                choices=list(QUESTION_BANK.keys()),
                value="Python",
                label="📚 Select Viva Subject"
            )

            number = gr.Slider(
                minimum=1,
                maximum=4,
                value=3,
                step=1,
                label="🔢 Number of Questions"
            )

            start_btn = gr.Button(
                "🚀 START VIVA",
                variant="primary"
            )

            reset_btn = gr.Button(
                "🔄 RESET"
            )

        with gr.Column(scale=2):

            question_display = gr.Markdown(
                "### 🎓 Viva not started"
            )

    gr.Markdown("---")

    with gr.Row():

        with gr.Column():

            gr.Markdown(
                "## 🎤 Your Answer"
            )

            audio = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="🎙️ Record / Upload Voice Answer"
            )

            answer = gr.Textbox(
                label="📝 Answer Transcript",
                placeholder=(
                    "Type your answer here, "
                    "or paste the transcript of your voice answer..."
                ),
                lines=8
            )

            sample_btn = gr.Button(
                "✨ Use Sample Answer"
            )

            submit_btn = gr.Button(
                "✅ SUBMIT ANSWER",
                variant="primary"
            )

        with gr.Column():

            feedback = gr.Markdown(
                "## 📊 Answer Evaluation\n"
                "Your evaluation will appear here."
            )

    gr.Markdown("---")

    next_display = gr.Markdown(
        "### 🎤 Next question will appear here."
    )

    status = gr.Textbox(
        label="System Status",
        interactive=False
    )

    gr.Markdown("---")

    report_btn = gr.Button(
        "📋 GENERATE FINAL REPORT",
        variant="primary"
    )

    final_report_output = gr.Markdown(
        "## 📋 Final Report\nComplete the viva first."
    )

    final_table = gr.Dataframe(
        label="📊 Question-wise Performance",
        interactive=False
    )

    # ========================================================
    # BUTTON CONNECTIONS
    # ========================================================

    start_btn.click(
        start_viva,
        inputs=[
            topic,
            number
        ],
        outputs=[
            question_display,
            status,
            feedback,
            next_display
        ]
    )

    sample_btn.click(
        sample_answer,
        outputs=answer
    )

    submit_btn.click(
        submit_answer,
        inputs=answer,
        outputs=[
            feedback,
            next_display,
            answer,
            status,
            gr.State()
        ]
    )

    report_btn.click(
        final_report,
        outputs=[
            final_report_output,
            final_table
        ]
    )

    reset_btn.click(
        reset_viva,
        outputs=[
            question_display,
            feedback,
            next_display,
            status,
            final_table
        ]
    )


# ============================================================
# LAUNCH
# ============================================================

print("=" * 60)
print("🎓 VOICE-BASED VIVA EXAMINER")
print("=" * 60)
print("Application starting...")
print("=" * 60)

app.launch(
    share=True,
    debug=False
)

/tmp/ipykernel_1488/3160544622.py:874: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


🎓 VOICE-BASED VIVA EXAMINER
Application starting...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6d7fce7b0eb2993a79.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
